# Data Partition cho Federated Learning

Notebook tạo file partition `.pt` phân chia dữ liệu MNIST hoặc CIFAR-10 cho các client, hỗ trợ 2 chế độ:

| `PARTITION_TYPE` | Mô tả |
|---|---|
| `dirichlet` | Non-IID theo phân phối Dirichlet(α). α nhỏ → skew cao, α lớn → gần IID. |
| `noniid_2`, `noniid_3`, … | Mỗi client nhận đúng **N class** (phân đều trong nội bộ class). Số sau `_` là số class mỗi client. |

`IMBALANCE_MODE = "Global"` sẽ lấy số mẫu theo `SAMPLES_PER_CLASS` trước, sau đó partition tập con như bình thường. Chỉ số client vẫn được lưu theo dataset gốc.

## ⚙️ Cấu hình

In [ ]:
# ── Tham số chính ──────────────────────────────────────────────────────────────
PARTITION_TYPE = "dirichlet"  # "dirichlet" | "noniid_2" | "noniid_3" | ...
DATASET_NAME   = "cifar10"    # "mnist" hoặc "cifar10"
IMBALANCE_MODE = "Global"     # "None" hoặc "Global": mất cân bằng class trước khi partition
NUM_CLIENTS    = 25
SEED           = 42
MIN_SAMPLES    = 100           # Số mẫu tối thiểu mỗi client (chỉ áp dụng Dirichlet)

# Tham số cho Dirichlet
ALPHA = 0.1                    # Concentration parameter

# Tham số tạo embedding ResNet-18 (dùng cho MODEL_NAME = 'resnet_linear')
GENERATE_EMBEDDINGS  = True
RESNET_SIZE          = 112
EMBED_BATCH_SIZE     = 256
OVERWRITE_EMBEDDINGS = False

# Số mẫu mỗi class khi IMBALANCE_MODE = "Global"
if DATASET_NAME.lower() == "mnist":
    SAMPLES_PER_CLASS = {
        0: 5900, 1: 5900, 2: 5300, 3: 4100, 4: 2900,
        5: 2100, 6: 1400, 7: 900, 8: 600, 9: 300,
    }
elif DATASET_NAME.lower() == "cifar10":
    SAMPLES_PER_CLASS = {
        0: 5000, 1: 5000, 2: 4500, 3: 3500, 4: 2500,
        5: 1800, 6: 1200, 7: 800, 8: 500, 9: 250,
    }
else:
    raise ValueError("DATASET_NAME phải là 'mnist' hoặc 'cifar10'.")

# ── Đường dẫn ─────────────────────────────────────────────────────────────────
PROJECT_ROOT = None            # None = thư mục hiện tại; hoặc đặt thành "/path/to/project"
OVERWRITE    = False           # Ghi đè file nếu đã tồn tại
NO_PLOT      = False           # True = bỏ qua bước vẽ biểu đồ

## 📦 Imports & Helpers

In [ ]:
import csv
import hashlib
import pickle
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, models


def _resolve_project_root(root_override) -> Path:
    """Trả về PROJECT_ROOT: ưu tiên tham số, sau đó Colab Drive, cuối cùng là CWD."""
    if root_override is not None:
        return Path(root_override)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/FLasIRL(new)")
    except ImportError:
        return Path.cwd()


def _load_cifar10_from_drive(data_dir: Path):
    """Đọc trực tiếp CIFAR-10 từ Drive, hoàn toàn không tự tải."""
    batch_dir = data_dir / "cifar-10-batches-py"
    if not batch_dir.is_dir():
        batch_dir = data_dir / "batches-py"
    if not batch_dir.is_dir():
        raise FileNotFoundError(f"CIFAR-10 batches-py không tồn tại trên Drive: {data_dir}")

    def read_batch(path: Path):
        with path.open("rb") as handle:
            batch = pickle.load(handle, encoding="bytes")
        images = torch.tensor(batch[b"data"]).reshape(-1, 3, 32, 32).float().div_(255.0)
        labels = torch.tensor(batch.get(b"labels", batch.get("labels")), dtype=torch.long)
        return images, labels

    train_parts = [read_batch(batch_dir / f"data_batch_{i}") for i in range(1, 6)]
    test_images, test_labels = read_batch(batch_dir / "test_batch")
    train_images = torch.cat([part[0] for part in train_parts])
    train_labels = torch.cat([part[1] for part in train_parts])
    return train_images, train_labels, test_images, test_labels


def _load_dataset_from_drive(dataset_name: str, data_dir: Path):
    """Tải dataset train & test có sẵn từ Drive, hoàn toàn không tự tải."""
    dataset_name = dataset_name.lower()
    if dataset_name == "mnist":
        train_ds = datasets.MNIST(data_dir, train=True, download=False)
        test_ds = datasets.MNIST(data_dir, train=False, download=False)
        train_images = train_ds.data.float().div_(255.0)
        test_images = test_ds.data.float().div_(255.0)
        train_labels = train_ds.targets.long()
        test_labels = test_ds.targets.long()
    elif dataset_name == "cifar10":
        train_images, train_labels, test_images, test_labels = _load_cifar10_from_drive(data_dir)
    else:
        raise ValueError(
            f"DATASET_NAME không hợp lệ: '{dataset_name}'. Dùng 'mnist' hoặc 'cifar10'."
        )
    return train_images, train_labels, test_images, test_labels


def _make_global_imbalance(
    train_labels: torch.Tensor,
    samples_per_class: dict[int, int],
    seed: int = 42,
) -> tuple[torch.Tensor, torch.Tensor, dict[int, int]]:
    """Lấy ngẫu nhiên số mẫu chỉ định của từng class trước khi partition."""
    labels_np = train_labels.cpu().numpy()
    classes = sorted(int(c) for c in np.unique(labels_np))
    requested_classes = set(int(c) for c in samples_per_class)
    if requested_classes != set(classes):
        raise ValueError(
            f"SAMPLES_PER_CLASS phải chứa đúng các class {classes}, "
            f"nhận được {sorted(requested_classes)}."
        )

    rng = np.random.default_rng(seed)
    selected = []
    actual_counts = {}
    for class_id in classes:
        target_count = int(samples_per_class[class_id])
        if target_count < 1:
            raise ValueError(f"Số mẫu class {class_id} phải >= 1.")
        class_indices = np.flatnonzero(labels_np == class_id)
        if target_count > len(class_indices):
            raise ValueError(
                f"Class {class_id} chỉ có {len(class_indices):,} mẫu, "
                f"không thể lấy {target_count:,} mẫu."
            )
        rng.shuffle(class_indices)
        selected.extend(class_indices[:target_count].tolist())
        actual_counts[class_id] = target_count

    rng.shuffle(selected)
    selected_indices = torch.tensor(selected, dtype=torch.long)
    selected_labels = train_labels[selected_indices]
    return selected_indices, selected_labels, actual_counts

def _parse_partition_type(partition_type: str) -> tuple[str, int | None]:
    """
    Parse PARTITION_TYPE:
      - "dirichlet"  → ("dirichlet", None)
      - "noniid_2"   → ("noniid", 2)
      - "noniid_10"  → ("noniid", 10)
    """
    if partition_type == "dirichlet":
        return "dirichlet", None
    m = re.fullmatch(r"noniid_(\d+)", partition_type)
    if m:
        return "noniid", int(m.group(1))
    raise ValueError(
        f"PARTITION_TYPE không hợp lệ: '{partition_type}'. "
        "Dùng 'dirichlet' hoặc 'noniid_N' (ví dụ: 'noniid_2', 'noniid_3')."
    )

## 🔧 Hàm tạo partition

In [ ]:
def generate_dirichlet_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    alpha: float = 0.5,
    seed: int = 42,
    min_samples: int = 10,
    max_retries: int = 100,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    """
    Phân chia chỉ số theo phân phối Dirichlet-Multinomial trên từng class.

    Returns:
        client_indices      : list[Tensor] – chỉ số mẫu của mỗi client.
        client_class_counts : Tensor (num_clients, num_classes) – số mẫu theo class.
    """
    labels_np   = train_labels.cpu().numpy()
    num_samples = len(labels_np)
    classes     = np.unique(labels_np)
    num_classes = len(classes)

    for attempt in range(max_retries):
        rng     = np.random.default_rng(seed + attempt)
        buckets = [[] for _ in range(num_clients)]

        for c in classes:
            idx_c       = np.where(labels_np == c)[0]
            rng.shuffle(idx_c)
            proportions = rng.dirichlet(np.full(num_clients, alpha))
            counts      = rng.multinomial(len(idx_c), proportions)
            start = 0
            for cid, cnt in enumerate(counts):
                if cnt > 0:
                    buckets[cid].extend(idx_c[start : start + cnt])
                    start += cnt

        if min(len(b) for b in buckets) >= min_samples:
            break
    else:
        raise RuntimeError(
            f"Không thể tạo partition Dirichlet (alpha={alpha}, min_samples={min_samples}) "
            f"sau {max_retries} lần thử. Hãy giảm min_samples hoặc tăng alpha."
        )

    client_indices      = []
    client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
    for cid, indices in enumerate(buckets):
        arr = np.array(indices, dtype=np.int64)
        rng.shuffle(arr)
        t   = torch.tensor(arr, dtype=torch.long)
        client_indices.append(t)
        client_class_counts[cid] = torch.bincount(train_labels[t], minlength=num_classes)

    assert sum(len(x) for x in client_indices) == num_samples, "Tổng mẫu không khớp!"
    assert len(torch.cat(client_indices).unique()) == num_samples, "Có mẫu bị trùng lặp!"
    return client_indices, client_class_counts

In [ ]:
def generate_noniid_partition(
    train_labels: torch.Tensor,
    num_clients: int = 25,
    classes_per_client: int = 2,
    seed: int = 42,
) -> tuple[list[torch.Tensor], torch.Tensor]:
    """
    Mỗi client nhận đúng `classes_per_client` class; mẫu được chia ngẫu nhiên
    không đều giữa các client cùng nhận một class.

    Returns:
        client_indices      : list[Tensor] – chỉ số mẫu của mỗi client.
        client_class_counts : Tensor (num_clients, num_classes) – số mẫu theo class.
    """
    labels_np   = train_labels.cpu().numpy()
    classes     = np.unique(labels_np)
    num_classes = len(classes)
    rng         = np.random.default_rng(seed)

    if not (1 <= classes_per_client <= num_classes):
        raise ValueError(
            f"classes_per_client phải nằm trong [1, {num_classes}], nhận được {classes_per_client}."
        )

    total_slots = num_clients * classes_per_client
    if total_slots % num_classes != 0:
        raise ValueError(
            "num_clients * classes_per_client phải chia hết cho số class "
            "để giữ số client nhận mỗi class cân bằng."
        )
    clients_per_class = total_slots // num_classes

    # Tạo nhóm class ngẫu nhiên. Với noniid_2, dùng các matching ngẫu nhiên
    # để mỗi class xuất hiện đúng số lần và không lặp cặp class.
    if classes_per_client == 2:
        for _ in range(1000):
            candidate_pairs = []
            for _ in range(clients_per_class):
                shuffled_classes = classes.copy()
                rng.shuffle(shuffled_classes)
                candidate_pairs.extend(
                    tuple(sorted((int(shuffled_classes[i]), int(shuffled_classes[i + 1]))))
                    for i in range(0, num_classes, 2)
                )
            if len(set(candidate_pairs)) == num_clients:
                client_classes = np.asarray(candidate_pairs, dtype=classes.dtype)
                break
        else:
            raise RuntimeError("Không tạo được các cặp class khác nhau sau 1000 lần thử.")
    else:
        # Trường hợp tổng quát: xáo trộn các slot class cân bằng và loại
        # các nhóm bị trùng class trong cùng một client.
        class_slots = np.repeat(classes, clients_per_class)
        for _ in range(1000):
            rng.shuffle(class_slots)
            candidate = class_slots.reshape(num_clients, classes_per_client)
            if all(len(set(row.tolist())) == classes_per_client for row in candidate):
                client_classes = candidate.copy()
                break
        else:
            raise RuntimeError("Không tạo được nhóm class hợp lệ sau 1000 lần thử.")

    rng.shuffle(client_classes, axis=0)

    buckets = [[] for _ in range(num_clients)]
    for class_id in classes:
        class_clients = np.where(client_classes == class_id)[0]
        class_indices = np.flatnonzero(labels_np == class_id)
        rng.shuffle(class_indices)
        if len(class_indices) < len(class_clients):
            raise ValueError(f"Class {class_id} không đủ mẫu để cấp cho các client.")
        # Chia ngẫu nhiên, không đều, nhưng mỗi client vẫn nhận ít nhất 1 mẫu
        # của class đã được gán.
        proportions = rng.dirichlet(np.ones(len(class_clients)))
        counts = rng.multinomial(len(class_indices) - len(class_clients), proportions) + 1
        start = 0
        for cid, count in zip(class_clients, counts):
            buckets[cid].extend(class_indices[start:start + count].tolist())
            start += count

    client_indices      = []
    client_class_counts = torch.zeros((num_clients, num_classes), dtype=torch.long)
    for cid, indices in enumerate(buckets):
        arr = np.asarray(indices, dtype=np.int64)
        rng.shuffle(arr)
        t   = torch.tensor(arr, dtype=torch.long)
        client_indices.append(t)
        client_class_counts[cid] = torch.bincount(train_labels[t], minlength=num_classes)

    assert sum(len(x) for x in client_indices) == len(labels_np), "Tổng mẫu không khớp!"
    assert len(torch.cat(client_indices).unique()) == len(labels_np), "Có mẫu bị trùng lặp!"
    assert torch.all((client_class_counts > 0).sum(dim=1) == classes_per_client), \
        "Có client không nhận đủ số class!"
    return client_indices, client_class_counts

## 📊 Hàm vẽ biểu đồ

In [ ]:
def plot_distribution(
    client_class_counts: torch.Tensor,
    title: str,
    chart_path: Path,
) -> None:
    """Vẽ stacked-bar chart phân bố class của các client và lưu ảnh."""
    num_clients  = client_class_counts.shape[0]
    num_classes  = client_class_counts.shape[1]
    client_sizes = client_class_counts.sum(dim=1).cpu().numpy()
    colors = plt.cm.tab10(np.linspace(0, 1, num_classes))

    fig, ax = plt.subplots(figsize=(max(10, num_clients // 2), 5))
    bottom  = np.zeros(num_clients)

    for cls in range(num_classes):
        vals = client_class_counts[:, cls].cpu().numpy()
        ax.bar(np.arange(num_clients), vals, bottom=bottom,
               label=f"Class {cls}", color=colors[cls],
               edgecolor="black", linewidth=0.4)
        bottom += vals

    ax.set_title(
        f"{title}\nMin={client_sizes.min()}, Max={client_sizes.max()}, Mean={client_sizes.mean():.1f}",
        fontsize=12,
    )
    ax.set_xlabel("Client ID", fontsize=11)
    ax.set_ylabel("Số mẫu", fontsize=11)
    ax.set_xticks(np.arange(num_clients))
    ax.set_xticklabels(range(num_clients), fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(ncol=num_classes, loc="upper center",
              bbox_to_anchor=(0.5, -0.18), frameon=True, fontsize=9)
    fig.tight_layout()

    chart_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Biểu đồ đã lưu: {chart_path}")

## 🚀 Chạy partition

In [ ]:
# ── Parse & validate cấu hình ────────────────────────────────────────────────
DATASET_NAME = DATASET_NAME.lower()
if DATASET_NAME not in {"mnist", "cifar10"}:
    raise ValueError("DATASET_NAME phải là 'mnist' hoặc 'cifar10'.")
imbalance_mode = str(IMBALANCE_MODE).strip().lower()
if imbalance_mode not in {"none", "global"}:
    raise ValueError("IMBALANCE_MODE phải là 'None' hoặc 'Global'.")
ptype, classes_per_client = _parse_partition_type(PARTITION_TYPE)

# ── Đường dẫn ─────────────────────────────────────────────────────────────────
project_root  = _resolve_project_root(PROJECT_ROOT)
data_dir      = project_root / "data" / DATASET_NAME
partition_dir = data_dir / "partitions" / "Dirichlet"
data_dir.mkdir(parents=True, exist_ok=True)
partition_dir.mkdir(parents=True, exist_ok=True)

if ptype == "dirichlet":
    stem = f"{DATASET_NAME}_dirichlet_alpha{ALPHA}_{NUM_CLIENTS}_seed{SEED}"
else:
    stem = f"noniid_{classes_per_client}_{NUM_CLIENTS}clients_seed{SEED}"
if imbalance_mode == "global":
    stem += "_global"

partition_path = partition_dir / f"{stem}.pt"
chart_path     = partition_dir / f"{stem}_dist.png"

print(f"Project root   : {project_root}")
print(f"Partition file : {partition_path}")

# ── Kiểm tra tồn tại ──────────────────────────────────────────────────────────
loaded_dataset = None
def _get_dataset():
    global loaded_dataset
    if loaded_dataset is None:
        print(f"\nĐang đọc {DATASET_NAME.upper()} từ Drive …")
        loaded_dataset = _load_dataset_from_drive(DATASET_NAME, data_dir)
    return loaded_dataset

if partition_path.exists() and not OVERWRITE:
    print("\n⚠️  File partition đã tồn tại. Đặt OVERWRITE = True để tạo lại.")
else:
    # ── Load dataset được chọn ────────────────────────────────────────────────
    _, train_labels, _, _ = _get_dataset()
    original_train_labels = train_labels
    original_num_samples = len(original_train_labels)
    if imbalance_mode == "global":
        selected_indices, partition_labels, global_class_counts = _make_global_imbalance(
            original_train_labels, SAMPLES_PER_CLASS, SEED
        )
        print(
            f"Tải xong: {original_num_samples:,} mẫu gốc; "
            f"Global imbalance còn {len(partition_labels):,} mẫu."
        )
        print("Số mẫu theo class:", global_class_counts)
    else:
        selected_indices = torch.arange(original_num_samples, dtype=torch.long)
        partition_labels = original_train_labels
        global_class_counts = {
            int(class_id): int((original_train_labels == class_id).sum())
            for class_id in torch.unique(original_train_labels)
        }
        print(f"Tải xong: {len(partition_labels):,} mẫu, {len(global_class_counts)} class.")
    num_classes = len(global_class_counts)

    # ── Tạo partition ─────────────────────────────────────────────────────────
    if ptype == "dirichlet":
        print(f"\nDirichlet: α={ALPHA}, K={NUM_CLIENTS}, seed={SEED}")
        client_indices, client_class_counts = generate_dirichlet_partition(
            partition_labels, NUM_CLIENTS, ALPHA, SEED, MIN_SAMPLES
        )
        chart_title = f"Dirichlet (α={ALPHA}, K={NUM_CLIENTS}, seed={SEED})"
        extra_meta  = {"alpha": ALPHA, "classes_per_client": None}

    else:  # noniid_N
        print(f"\n{PARTITION_TYPE}: {classes_per_client} class/client, K={NUM_CLIENTS}, seed={SEED}")
        client_indices, client_class_counts = generate_noniid_partition(
            partition_labels, NUM_CLIENTS, classes_per_client, SEED
        )
        chart_title = f"Non-IID {classes_per_client} class/client (K={NUM_CLIENTS}, seed={SEED})"
        extra_meta  = {"alpha": None, "classes_per_client": classes_per_client}

    # Đổi chỉ số cục bộ của tập con về chỉ số gốc của dataset.
    client_indices = [selected_indices[indices] for indices in client_indices]

    # ── Lưu partition ─────────────────────────────────────────────────────────
    client_sizes = [len(x) for x in client_indices]
    torch.save(
        {
            "client_indices" : client_indices,
            "class_counts"   : client_class_counts,
            "client_sizes"   : client_sizes,
            "distribution"   : PARTITION_TYPE,
            "dataset_name"   : DATASET_NAME,
            "imbalance_mode" : "Global" if imbalance_mode == "global" else "None",
            "global_class_counts": global_class_counts,
            "samples_per_class": SAMPLES_PER_CLASS if imbalance_mode == "global" else None,
            "num_clients"    : NUM_CLIENTS,
            "seed"           : SEED,
            **extra_meta,
        },
        partition_path,
    )
    print(f"\n✅ Đã lưu: {partition_path}")

    # ── Thống kê ──────────────────────────────────────────────────────────────
    print(f"\n{'─'*42}")
    print(f"  Partition     : {PARTITION_TYPE}")
    print(f"  Num clients   : {NUM_CLIENTS}")
    print(f"  Total samples : {sum(client_sizes):,}")
    print(f"  Min / Max / Mean : {min(client_sizes)} / {max(client_sizes)} / {np.mean(client_sizes):.1f}")
    print(f"  Std           : {np.std(client_sizes):.1f}")
    print(f"{'─'*42}")

    # ── Vẽ biểu đồ ───────────────────────────────────────────────────────────
    if not NO_PLOT:
        plot_distribution(client_class_counts, chart_title, chart_path)

# ── Tạo & cache ResNet-18 embeddings (dùng chung cho 4 notebook train) ────────
def embedding_cache_filename(dataset_name: str, resnet_size: int, split: str) -> str:
    key = f"{dataset_name}_resnet18_imagenet1k_v1_resize{resnet_size}_imagenet_norm_{split}"
    h = hashlib.sha256(key.encode()).hexdigest()[:8]
    return f"resnet18_sz{resnet_size}_{split}_{h}.pt"

if GENERATE_EMBEDDINGS:
    train_embed_path = data_dir / embedding_cache_filename(DATASET_NAME, RESNET_SIZE, "train")
    test_embed_path = data_dir / embedding_cache_filename(DATASET_NAME, RESNET_SIZE, "test")
    if train_embed_path.exists() and test_embed_path.exists() and not OVERWRITE_EMBEDDINGS:
        print(f"\n✅ Embeddings đã tồn tại:\n  - {train_embed_path}\n  - {test_embed_path}")
    else:
        embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"\nĐang tạo ResNet-18 embeddings (size={RESNET_SIZE}) trên {embed_device} …")
        train_images, _, test_images, _ = _get_dataset()
        imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=embed_device).view(1, 3, 1, 1)
        imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=embed_device).view(1, 3, 1, 1)

        def preprocess_resnet(images):
            x = images.unsqueeze(1).repeat(1, 3, 1, 1) if DATASET_NAME == "mnist" else images
            x = F.interpolate(x, size=(RESNET_SIZE, RESNET_SIZE), mode="bilinear", align_corners=False)
            return (x - imagenet_mean) / imagenet_std

        def make_embeddings(images, backbone):
            backbone.eval()
            output = []
            with torch.no_grad():
                for start in range(0, len(images), EMBED_BATCH_SIZE):
                    batch = preprocess_resnet(images[start:start + EMBED_BATCH_SIZE].to(embed_device))
                    output.append(backbone(batch).float().cpu())
            return torch.cat(output)

        weights = models.ResNet18_Weights.IMAGENET1K_V1
        backbone = models.resnet18(weights=weights).to(embed_device)
        backbone.fc = nn.Identity()
        for parameter in backbone.parameters():
            parameter.requires_grad_(False)
        train_features = make_embeddings(train_images, backbone)
        test_features = make_embeddings(test_images, backbone)
        torch.save(train_features, train_embed_path)
        torch.save(test_features, test_embed_path)
        del backbone
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ Đã lưu embeddings:\n  - {train_embed_path} {tuple(train_features.shape)}\n  - {test_embed_path} {tuple(test_features.shape)}")

## 💾 Xuất CSV thống kê class

In [ ]:
partition   = torch.load(partition_path, map_location="cpu", weights_only=False)
counts_mat  = partition["class_counts"]     # (num_clients, num_classes)
num_classes = counts_mat.shape[1]
csv_path    = partition_path.with_name(partition_path.stem + "_class_counts.csv")

fieldnames = ["client_id", "total_samples"] + [f"class_{c}" for c in range(num_classes)]

with csv_path.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    for cid, row in enumerate(counts_mat.tolist()):
        writer.writerow({
            "client_id"     : cid,
            "total_samples" : sum(row),
            **{f"class_{c}": v for c, v in enumerate(row)},
        })

print(f"CSV đã lưu: {csv_path}")